# 

In [1]:
from ogb.utils.features import (allowable_features, atom_to_feature_vector,
 bond_to_feature_vector, atom_feature_vector_to_dict, bond_feature_vector_to_dict) 
import numpy as np
from tqdm import tqdm
import instructions_smol
import datasets
from datasets import load_dataset
import pandas as pd
import os
from rdkit import Chem
from ood_dataset_download import get_data_list, system_prompt
import selfies as sf

/miniconda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
mol_instruction_dataset = datasets.load_dataset(
    "zjunlp/Mol-Instructions",
    "Molecule-oriented Instructions",
    trust_remote_code=True,
)
qm9_data = mol_instruction_dataset['property_prediction']
qm9_homo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo
            )
qm9_lumo_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_lumo
            )
qm9_homo_lumo_gap_data = qm9_data.filter(
                lambda x: x["instruction"] in instructions_smol.filtering_template_homo_lumo_gap
            )

Filter: 100%|██████████| 362100/362100 [00:02<00:00, 148278.89 examples/s]


In [3]:
def get_qm9_data_list(
        qm9_data,
        task,
        instruction_templates,
):
    list_tr_mol = []
    list_tr_label = []

    list_te_mol = []
    list_te_label = []

    from tqdm import tqdm
    iter_bar = tqdm(range(len(qm9_data)))
    for i in iter_bar:
        data_instance = qm9_data[i]
        selfies = data_instance['input']
        smiles = sf.decoder(selfies)
        mol = Chem.MolFromSmiles(smiles)
        label = data_instance['output']

        if "train" in data_instance['metadata']:
            list_tr_label.append(label)
            list_tr_mol.append(mol)
        elif "test" in data_instance['metadata']:
            list_te_label.append(label)
            list_te_mol.append(mol)
        elif "val" in data_instance['metadata']:
            pass
        else:
            print(data_instance)
            raise ValueError

    list_tr_data = get_data_list(
        list_mol=list_tr_mol,
        list_label=list_tr_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    list_te_data = get_data_list(
        list_mol=list_te_mol,
        list_label=list_te_label,
        task=task,
        instruction_templates=instruction_templates,
    )
    print(len(list_tr_data), len(list_te_data))
    return list_tr_data, list_te_data

In [9]:
list_qm9_homo_tr_data, list_qm9_homo_te_data = get_qm9_data_list(
    qm9_data=qm9_homo_data,
    instruction_templates=instructions_smol.qm9_homo,
    task="qm9_homo",
)

100%|██████████| 684/684 [00:00<00:00, 1974.16it/s]

120062 684


In [10]:
homo_dict = {
    "train": list_qm9_homo_tr_data,
    "test": list_qm9_homo_te_data
}


for data_dict in [
    homo_dict
]:
    for split in ["train", "test"]:
        list_data = data_dict[split]
        
        dataset = datasets.Dataset.from_list(list_data)
        dataset.save_to_disk(
            f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_qm9_homo_0219"
        )

Saving the dataset (1/1 shards): 100%|██████████| 684/684 [00:00<00:00, 14684.84 examples/s]


In [5]:
list_qm9_lumo_tr_data, list_qm9_lumo_te_data = get_qm9_data_list(
    qm9_data=qm9_lumo_data,
    instruction_templates=instructions_smol.qm9_lumo,
    task="qm9_lumo",
)

lumo_dict = {
    'task_name': 'lumo',
    "train": list_qm9_lumo_tr_data,
    "test": list_qm9_lumo_te_data
}

list_qm9_homo_lumo_gap_tr_data, list_qm9_homo_lumo_gap_te_data = get_qm9_data_list(
    qm9_data=qm9_homo_lumo_gap_data,
    instruction_templates=instructions_smol.qm9_homo_lumo_gap,
    task="qm9_homo_lumo_gap",
)

gap_dict = {
    'task_name': 'homo_lumo_gap',
    "train": list_qm9_homo_lumo_gap_tr_data,
    "test": list_qm9_homo_lumo_gap_te_data
}


for data_dict in [
    lumo_dict,
    gap_dict
]:
    for split in ["train", "test"]:
        list_data = data_dict[split]
        task_name = data_dict['task_name']
        dataset = datasets.Dataset.from_list(list_data)
        dataset.save_to_disk(
            f"/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_{split}_qm9_{task_name}_0219"
        )

100%|██████████| 642/642 [00:00<00:00, 1974.24it/s]


120111 642


100%|██████████| 661/661 [00:00<00:00, 1969.38it/s]


119940 661


Saving the dataset (1/1 shards): 100%|██████████| 661/661 [00:00<00:00, 14633.97 examples/s]


In [6]:
train_pcqm_paths = [
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_lumo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_lumo_gap_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_pcqm_homo_lumo_gap_0219",
]
test_paths = [
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_homo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_lumo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_homo_lumo_gap_0219",
]

In [7]:
train_pcqm_paths = [
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_lumo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_lumo_gap_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_pcqm_homo_lumo_gap_0219",
]
test_paths = [
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_homo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_lumo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_homo_lumo_gap_0219",
]

train_pcqm_data = []
for i in train_pcqm_paths:
    loaded_data = datasets.load_from_disk(i)
    train_pcqm_data.append(loaded_data)

test_data = []
for i in test_paths:
    loaded_data = datasets.load_from_disk(i)
    test_data.append(loaded_data)

In [13]:
# concatenate dataset
concated_train_pcqm_dataset = datasets.concatenate_datasets(train_pcqm_data)
concated_train_pcqm_dataset.save_to_disk(
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_pcqm_augmented_0219",
)
concated_test_dataset = datasets.concatenate_datasets(test_data)
concated_test_dataset.save_to_disk(
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_test_qm9_pcqm_augmented_0219",
)


Saving the dataset (1/1 shards): 100%|██████████| 1987/1987 [00:00<00:00, 20275.80 examples/s]


In [15]:
train_qm9_paths = [
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_lumo_0219",
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_homo_lumo_gap_0219",
]

train_qm9_data = []
for i in train_qm9_paths:
    loaded_data = datasets.load_from_disk(i)
    train_qm9_data.append(loaded_data)

# concatenate dataset
concated_train_qm9_dataset = datasets.concatenate_datasets(train_qm9_data)
concated_train_qm9_dataset.save_to_disk(
    "/data/data/Mol-LLM-v7.1/mistralai-Mistral-7B-Instruct-v0.3_string+graph_q32_train_qm9_0219",
)

Saving the dataset (3/3 shards): 100%|██████████| 360113/360113 [00:12<00:00, 29720.12 examples/s] 
